# Customer Churn Prediction using Machine Learning

Customer churn refers to the situation when customers stop using a company's product or service. For businesses such as telecom companies, banks, and subscription-based platforms, understanding and predicting churn is extremely important because retaining existing customers is usually more cost-effective than acquiring new ones.

In this project, we build a **Customer Churn Prediction Model** using machine learning techniques. The goal is to analyze customer data and identify patterns that indicate whether a customer is likely to leave the service.

The main objectives of this notebook are:

* Perform **data exploration and preprocessing** to understand the dataset
* Conduct **Exploratory Data Analysis (EDA)** to discover important patterns
* Apply **feature engineering and data cleaning**
* Train and evaluate **machine learning models** for churn prediction
* Analyze the model performance and identify key factors contributing to churn

This project demonstrates a complete **end-to-end machine learning workflow**, from raw data analysis to predictive modeling, which can help organizations make data-driven decisions to improve customer retention.

Data from https://www.kaggle.com/blastchar/telco-customer-churn


## 1. Import Required Libraries

In this step, we import the essential Python libraries that will be used throughout the notebook for data manipulation, analysis, and visualization.

* **Pandas** is used for loading and manipulating structured data using DataFrames.
* **NumPy** provides support for numerical operations and efficient array computations.
* **Seaborn** is a statistical data visualization library that helps create informative and attractive plots.
* **Matplotlib** is a core plotting library used for creating various types of charts and graphs.

The `%matplotlib inline` command ensures that all visualizations generated using Matplotlib are displayed directly within the notebook.

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

## 2. Load the Dataset

In this step, we load the **Telco Customer Churn dataset** into the notebook using Pandas.

The dataset contains information about telecom customers, including their demographic details, account information, services they use, and whether they have churned (left the company) or not.

Using `pandas.read_csv()`, the dataset is read from a CSV file and stored in a DataFrame named **df**, which will be used throughout the analysis and model development process.


In [2]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

## 3. Check Dataset Size

In this step, we check the total number of records in the dataset using the `len()` function.

This helps us understand how many customer entries are present in the dataset. Knowing the dataset size is useful before performing further analysis, preprocessing, and model training.


In [3]:
len(df)

7043

## 4. Initial data preparation

## Preview the Dataset

In this step, we display the first few rows of the dataset using the `head()` function.

This allows us to quickly inspect the structure of the data, including the column names, data types, and example values. It helps in gaining an initial understanding of the dataset before performing further data cleaning and analysis.

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Transpose Preview of the Dataset

Here, we use the `T` attribute to **transpose the first few rows** of the dataset.

Transposing the dataset flips rows and columns, making it easier to view column names along with their corresponding values vertically. This is especially useful when the dataset has many columns and a horizontal preview is hard to read.


In [5]:
df.head().T

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


## Check Data Types of Columns

In this step, we inspect the data types of each column in the dataset using `dtypes`.

Knowing the data types helps us understand which columns are **numerical** and which are **categorical**. This is important for data preprocessing, feature engineering, and selecting the appropriate machine learning models.

In [6]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

## Convert `TotalCharges` to Numeric and Handle Missing Values

The `TotalCharges` column may contain non-numeric values due to data entry issues or empty strings. Machine learning models require numeric input, so we perform the following steps:

1. **Convert to numeric**: Using `pd.to_numeric()` with `errors='coerce'`, any invalid or non-numeric entries are converted to `NaN`.
2. **Handle missing values**: Replace all `NaN` values with `0` using `fillna(0)` to ensure the column is fully numeric and ready for analysis.

This ensures that the `TotalCharges` column can be used safely in calculations and model training.

In [6]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

## Standardize Column Names and String Values

To make the dataset consistent and easier to work with, we perform the following cleaning steps:

1. **Column names**: Convert all column names to lowercase and replace spaces with underscores (`_`).
   This avoids issues when referencing columns in code.
2. **String values**: Identify all columns with `object` data type (categorical/text columns) and standardize their values by converting them to lowercase and replacing spaces with underscores.

Standardizing both column names and string values helps prevent errors during analysis, visualization, and machine learning tasks.

In [8]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

string_columns = list(df.dtypes[df.dtypes == 'object'].index)

for col in string_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_')

## Encode Target Variable

The `churn` column indicates whether a customer has left the company. Currently, it contains string values `'yes'` and `'no'`.

To prepare it for machine learning:

* Convert `'yes'` to `1` and `'no'` to `0` using a boolean comparison `(df.churn == 'yes')`.
* Cast the result to `int` so that the `churn` column becomes a **binary numeric variable**.

This ensures that the target variable is in the proper format for model training and evaluation.


In [9]:
df.churn = (df.churn == 'yes').astype(int)

## Verify Data Cleaning and Encoding

After performing data cleaning and encoding steps, we preview the first few rows of the dataset in a **transposed format** using `df.head().T`.

Transposing the dataset allows us to easily inspect each column vertically, confirming that:

* Column names are standardized (lowercase, underscores)
* String values are cleaned
* `TotalCharges` is numeric
* `churn` target variable is properly encoded as 0/1

This helps ensure the dataset is ready for further analysis and model training.

In [10]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


## 5. Import Train-Test Split Function

In this step, we import the `train_test_split` function from **Scikit-learn**.

This function is used to split the dataset into:

* **Training set**: Used to train the machine learning model.
* **Testing set**: Used to evaluate the model's performance on unseen data.

Splitting the data ensures that we can assess how well the model generalizes to new, unseen customers, which is crucial for reliable churn prediction.


In [11]:
from sklearn.model_selection import train_test_split

## 6. Split Dataset into Training and Testing Sets

Here, we split the dataset into **training** and **testing** sets using `train_test_split`:

* **df_train_full**: Contains 80% of the data, used for training and validation.
* **df_test**: Contains 20% of the data, reserved for final model evaluation.

The `test_size=0.2` parameter ensures 20% of the dataset is held out for testing, and `random_state=1` guarantees reproducibility of the split.

This separation allows us to train models on one portion of the data and evaluate their performance on unseen data, which is critical for reliable churn prediction.


In [12]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=1)

## 7. Create Training and Validation Sets

To optimize and evaluate the model before final testing, we further split the **training set** into:

* **df_train**: Used to train the machine learning model.
* **df_val**: Used as a validation set for model tuning and performance monitoring.

Here, `test_size=0.33` means approximately 33% of the original training data is reserved for validation, and `random_state=11` ensures reproducibility.

Using a separate validation set helps prevent overfitting and allows us to fine-tune the model before evaluating it on the unseen test set.


In [13]:
df_train, df_val = train_test_split(df_train_full, test_size=0.33, random_state=11)

## 8. Extract Target Variable

In this step, we separate the **target variable** `churn` from the training and validation sets:

* **y_train**: Contains the target values for the training set.
* **y_val**: Contains the target values for the validation set.

Using `.values` converts the target column into a NumPy array, which is the format expected by most machine learning models in Scikit-learn.

This separation ensures that the features and labels are ready for model training and evaluation.


In [14]:
y_train = df_train.churn.values
y_val = df_val.churn.values

## 9. Remove Target Column from Feature Sets

After extracting the target variable, we remove the `churn` column from the training and validation DataFrames:

* This ensures that **df_train** and **df_val** contain only the independent variables (features).
* Prevents accidental data leakage of the target into the model inputs during training.

Now, the feature sets are ready for preprocessing and model training.


In [15]:
del df_train['churn']
del df_val['churn']

## 10. Exploratory data analysis

## Check for Missing Values

In this step, we inspect the full training dataset for any **missing or null values** using `isnull().sum()`.

Identifying missing values is crucial because:

* They can cause errors during model training.
* Handling them appropriately (e.g., imputation or removal) improves model performance.

This check ensures that our dataset is complete and ready for preprocessing and feature engineering.

In [16]:
df_train_full.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

## Inspect Target Variable Distribution

Here, we examine the distribution of the target variable `churn` in the full training dataset using `value_counts()`.

* This shows how many customers have churned (`1`) versus those who stayed (`0`).
* Understanding class balance is important for model training, as imbalanced classes may require special handling (e.g., resampling, class weighting).

This step helps us assess whether the dataset is balanced and ready for model development.


In [17]:
df_train_full.churn.value_counts()

0    4113
1    1521
Name: churn, dtype: int64

## Calculate Overall Churn Rate

In this step, we compute the **global mean** of the target variable `churn`:

* `global_mean = df_train_full.churn.mean()` calculates the proportion of customers who have churned.
* `round(global_mean, 3)` rounds the result to three decimal places for readability.

This gives us the **overall churn rate**, which serves as a baseline metric for understanding customer behavior and for comparing model performance against a simple baseline prediction.


In [18]:
global_mean = df_train_full.churn.mean()
round(global_mean, 3)

0.27

In [19]:
categorical = ['gender', 'seniorcitizen', 'partner', 'dependents',
               'phoneservice', 'multiplelines', 'internetservice',
               'onlinesecurity', 'onlinebackup', 'deviceprotection',
               'techsupport', 'streamingtv', 'streamingmovies',
               'contract', 'paperlessbilling', 'paymentmethod']
numerical = ['tenure', 'monthlycharges', 'totalcharges']

## Define Categorical and Numerical Features

To prepare the dataset for analysis and modeling, we separate the features into two types:

* **Categorical features**: Variables representing categories or discrete values, such as `gender`, `contract`, and `paymentmethod`.
* **Numerical features**: Continuous variables, such as `tenure`, `monthlycharges`, and `totalcharges`.

Separating features by type helps in applying appropriate preprocessing techniques, such as encoding categorical variables and scaling numerical variables.


In [20]:
df_train_full[categorical].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

## 11. Feature importance

## Analyze Churn Rate by Gender

In this step, we calculate the mean churn rate separately for **female** and **male** customers:

* `female_mean` computes the proportion of female customers who have churned.
* `male_mean` computes the proportion of male customers who have churned.
* Using `round(..., 3)` rounds the values for readability.

This analysis helps identify potential differences in churn behavior between genders, providing initial insights for feature importance and business decisions.


In [21]:
female_mean = df_train_full[df_train_full.gender == 'female'].churn.mean()
print('gender == female:', round(female_mean, 3))

male_mean = df_train_full[df_train_full.gender == 'male'].churn.mean()
print('gender == male:  ', round(male_mean, 3))

gender == female: 0.277
gender == male:   0.263


## Compare Female Churn Rate to Overall Churn Rate

Here, we calculate the ratio of the **female churn rate** to the **overall churn rate**:

* `female_mean / global_mean` indicates how the churn behavior of female customers compares to the average churn in the dataset.
* A ratio greater than 1 means females churn at a higher rate than average, while a ratio less than 1 means lower than average.

This relative comparison provides insights into group-specific churn tendencies, which can inform targeted business strategies.


In [22]:
female_mean / global_mean

1.0253955354648652

## Compare Male Churn Rate to Overall Churn Rate

Here, we calculate the ratio of the **male churn rate** to the **overall churn rate**:

* `male_mean / global_mean` shows how the churn behavior of male customers compares to the average churn in the dataset.
* A ratio greater than 1 indicates males churn more than average, while a ratio less than 1 indicates they churn less.

This relative analysis helps identify whether gender plays a role in customer churn and can guide targeted retention strategies.


In [23]:
male_mean / global_mean

0.9749802969838747

## Analyze Churn Rate by Partner Status

In this step, we calculate the mean churn rate for customers based on their **partner status**:

* `partner_yes` computes the proportion of customers with a partner who have churned.
* `partner_no` computes the proportion of customers without a partner who have churned.
* Using `round(..., 3)` makes the values easy to interpret.

This analysis helps identify whether having a partner is associated with higher or lower likelihood of churn, providing insights into customer demographics and behavior.


In [24]:
partner_yes = df_train_full[df_train_full.partner == 'yes'].churn.mean()
print('partner == yes:', round(partner_yes, 3))

partner_no = df_train_full[df_train_full.partner == 'no'].churn.mean()
print('partner == no :', round(partner_no, 3))

partner == yes: 0.205
partner == no : 0.33


## Compare Churn Rate of Customers with Partner to Overall

Here, we calculate the ratio of the churn rate for customers with a partner to the overall churn rate:

* `partner_yes / global_mean` indicates whether having a partner increases or decreases the likelihood of churn compared to the average customer.
* A ratio greater than 1 means customers with a partner churn more than average, while a ratio less than 1 means they churn less.

This insight helps understand demographic influence on churn behavior.


In [25]:
partner_yes / global_mean

0.7594724924338315

## Compare Churn Rate of Customers without Partner to Overall

Here, we calculate the ratio of the churn rate for customers without a partner to the overall churn rate:

* `partner_no / global_mean` indicates whether customers without a partner are more or less likely to churn compared to the average customer.
* A ratio greater than 1 means customers without a partner churn more than average, while a ratio less than 1 means they churn less.

This relative analysis provides insights into how partnership status influences customer churn behavior.


In [26]:
partner_no / global_mean

1.2216593879412643

## Summary of Churn Analysis by Gender

In this step, we create a summary table to analyze churn behavior by gender:

* `mean`: Average churn rate for each gender.
* `diff`: Difference between the gender-specific churn rate and the overall churn rate (`mean - global_mean`).
* `risk`: Relative risk of churn compared to the overall churn rate (`mean / global_mean`).

This table provides a concise view of how gender impacts customer churn, highlighting which group is more likely to churn and by how much relative to the overall population.


In [27]:
df_group = df_train_full.groupby(by='gender').churn.agg(['mean'])
df_group['diff'] = df_group['mean'] - global_mean
df_group['risk'] = df_group['mean'] / global_mean
df_group

,mean,diff,risk
gender,,,
female,0.276824,0.006856,1.025396
male,0.263214,-0.006755,0.974980


## Import Display Function for Better DataFrame Visualization

Here, we import the `display` function from **IPython.display**:

* `display()` provides a cleaner and more readable way to show pandas DataFrames in Jupyter notebooks.
* It is especially useful when showing multiple DataFrames or summary tables, as it formats them neatly for easier inspection.

This enhances the visual presentation of data during exploratory analysis.


In [28]:
from IPython.display import display

## Recalculate Overall Churn Rate

In this step, we compute the **overall churn rate** of the full training dataset:

* `global_mean = df_train_full.churn.mean()` calculates the proportion of customers who have churned.
* Displaying `global_mean` provides a baseline reference for comparing group-specific churn rates and calculating relative risks.

Having this baseline helps contextualize subsequent analysis and insights.


In [29]:
global_mean = df_train_full.churn.mean()
global_mean

0.26996805111821087

## Churn Analysis for All Categorical Features

In this step, we analyze the impact of each categorical feature on churn:

* For each categorical column, we calculate:

  * **mean**: Average churn rate for each category.
  * **diff**: Difference from the overall churn rate (`mean - global_mean`).
  * **risk**: Relative risk compared to the overall churn rate (`mean / global_mean`).

* `display(df_group)` is used to neatly show each summary table in the notebook.

This analysis provides a clear view of how different customer categories influence churn, helping identify high-risk groups for targeted retention strategies.


In [30]:
for col in categorical:
    df_group = df_train_full.groupby(by=col).churn.agg(['mean'])
    df_group['diff'] = df_group['mean'] - global_mean
    df_group['risk'] = df_group['mean'] / global_mean
    display(df_group)

,mean,diff,risk
gender,,,
female,0.276824,0.006856,1.025396
male,0.263214,-0.006755,0.974980


,mean,diff,risk
seniorcitizen,,,
0,0.242270,-0.027698,0.897403
1,0.413377,0.143409,1.531208


,mean,diff,risk
partner,,,
no,0.329809,0.059841,1.221659
yes,0.205033,-0.064935,0.759472


,mean,diff,risk
dependents,,,
no,0.313760,0.043792,1.162212
yes,0.165666,-0.104302,0.613651


,mean,diff,risk
phoneservice,,,
no,0.241316,-0.028652,0.893870
yes,0.273049,0.003081,1.011412


...

,mean,diff,risk
streamingtv,,,
no,0.342832,0.072864,1.269897
no_internet_service,0.077805,-0.192163,0.288201
yes,0.302723,0.032755,1.121328


,mean,diff,risk
streamingmovies,,,
no,0.338906,0.068938,1.255358
no_internet_service,0.077805,-0.192163,0.288201
yes,0.307273,0.037305,1.138182


,mean,diff,risk
contract,,,
month-to-month,0.431701,0.161733,1.599082
one_year,0.120573,-0.149395,0.446621
two_year,0.028274,-0.241694,0.104730


,mean,diff,risk
paperlessbilling,,,
no,0.172071,-0.097897,0.637375
yes,0.338151,0.068183,1.252560


,mean,diff,risk
paymentmethod,,,
bank_transfer_(automatic),0.168171,-0.101797,0.622928
credit_card_(automatic),0.164339,-0.105630,0.608733
electronic_check,0.455890,0.185922,1.688682
mailed_check,0.193870,-0.076098,0.718121


## 12. Import Mutual Information Function

Here, we import `mutual_info_score` from **Scikit-learn**:

* Mutual information measures the **dependency between two variables**.
* It is particularly useful for **categorical features**, indicating how much information a feature provides about the target variable (`churn`).
* This metric can guide **feature selection** by highlighting features that are most relevant for predicting churn.


In [31]:
from sklearn.metrics import mutual_info_score

## Calculate Mutual Information for Categorical Features

In this step, we measure the **mutual information (MI)** between each categorical feature and the target variable `churn`:

* `calculate_mi(series)` applies `mutual_info_score` to each categorical column.

* Higher MI values indicate that the feature provides more information about the likelihood of churn.

* Sorting the MI values helps identify the **most and least informative features**.

* `display(df_mi.head())` shows the top features with the highest mutual information.

* `display(df_mi.tail())` shows the features with the lowest mutual information.

This analysis helps guide **feature selection** and prioritizes which categorical variables are most relevant for modeling.

In [32]:
def calculate_mi(series):
    return mutual_info_score(series, df_train_full.churn)

df_mi = df_train_full[categorical].apply(calculate_mi)
df_mi = df_mi.sort_values(ascending=False).to_frame(name='MI')


display(df_mi.head())
display(df_mi.tail())

,MI
contract,0.098320
onlinesecurity,0.063085
techsupport,0.061032
internetservice,0.055868
onlinebackup,0.046923


,MI
partner,0.009968
seniorcitizen,0.009410
multiplelines,0.000857
phoneservice,0.000229
gender,0.000117


## 13. Correlation of Numerical Features with Churn

In this step, we calculate the **correlation between each numerical feature and the target variable `churn`**:

* `corrwith()` computes the Pearson correlation coefficient for each numerical column.
* Positive correlation indicates that higher values of the feature are associated with higher likelihood of churn, while negative correlation indicates the opposite.
* Converting the result to a DataFrame with `.to_frame('correlation')` makes it easier to inspect.

This analysis helps identify which numerical features are most influential in predicting customer churn.


In [33]:
df_train_full[numerical].corrwith(df_train_full.churn).to_frame('correlation')

,correlation
tenure,-0.351885
monthlycharges,0.196805
totalcharges,-0.196353


## Compare Numerical Features by Churn Status

In this step, we calculate the **mean values of numerical features** for customers who **churned** (`churn = 1`) versus those who **did not churn** (`churn = 0`):

* `groupby('churn')[numerical].mean()` computes the class-wise averages.
* Comparing these means highlights differences in customer behavior, such as tenure, monthly charges, or total charges.
* This information can provide insights into which numerical features are more relevant for predicting churn.


In [34]:
df_train_full.groupby(by='churn')[numerical].mean()

,tenure,monthlycharges,totalcharges
churn,,,
0,37.531972,61.176477,2548.021627
1,18.070348,74.521203,1545.689415


## 14. One-hot encoding

## Import DictVectorizer for Encoding Categorical Features

Here, we import `DictVectorizer` from **Scikit-learn**:

* `DictVectorizer` converts **categorical variables** into a **numeric feature matrix** using one-hot encoding.
* It is particularly useful when working with **dictionary-like data** or pandas DataFrames for machine learning models.
* This step ensures that all categorical features are properly encoded and can be fed into algorithms that require numerical input.


In [35]:
from sklearn.feature_extraction import DictVectorizer

## Convert Training Features to Dictionary Format

In this step, we prepare the training data for encoding:

* `df_train[categorical + numerical].to_dict(orient='records')` converts the selected features into a **list of dictionaries**, where each dictionary represents a single training example.
* This format is compatible with `DictVectorizer`, allowing us to efficiently encode both categorical and numerical features into a numeric feature matrix.
* Preparing the data in this way ensures that all features are ready for model training.


In [36]:
train_dict = df_train[categorical + numerical].to_dict(orient='records')

## Inspect First Training Record in Dictionary Format

Here, we display the **first training example** after converting the dataset to a list of dictionaries:

* `train_dict[0]` shows the feature names as keys and their corresponding values.
* This verification step ensures that both categorical and numerical features have been correctly transformed and are ready for encoding with `DictVectorizer`.


In [37]:
train_dict[0]

{'gender': 'male',
 'seniorcitizen': 0,
 'partner': 'yes',
 'dependents': 'no',
 'phoneservice': 'yes',
 'multiplelines': 'no',
 'internetservice': 'dsl',
 'onlinesecurity': 'yes',
 'onlinebackup': 'yes',
 'deviceprotection': 'yes',
 'techsupport': 'yes',
 'streamingtv': 'yes',
 'streamingmovies': 'yes',
 'contract': 'two_year',
 'paperlessbilling': 'yes',
 'paymentmethod': 'bank_transfer_(automatic)',
 'tenure': 71,
 'monthlycharges': 86.1,
 'totalcharges': 6045.9}

## Fit DictVectorizer on Training Data

In this step, we initialize and fit a **DictVectorizer**:

* `DictVectorizer(sparse=False)` creates an encoder that will transform dictionary data into a numeric feature matrix.
* `dv.fit(train_dict)` learns the **encoding mapping** from the training data, including all unique values of categorical features.
* Fitting the vectorizer prepares it to convert both training and future datasets into a consistent numeric format for model training.


In [38]:
dv = DictVectorizer(sparse=False)
dv.fit(train_dict)

DictVectorizer(sparse=False)

## Transform Training Data into Feature Matrix

In this step, we use the fitted `DictVectorizer` to convert the training data into a **numeric feature matrix**:

* `X_train = dv.transform(train_dict)` applies the encoding learned during fitting.
* The resulting `X_train` contains both **one-hot encoded categorical features** and **numerical features** in a single matrix.
* This numeric matrix is now ready to be fed into machine learning algorithms for model training.


In [39]:
X_train = dv.transform(train_dict)

## Inspect Shape of Training Feature Matrix

Here, we check the dimensions of the transformed training data:

* `X_train.shape` returns the number of **rows (samples)** and **columns (features)** in the feature matrix.
* Verifying the shape ensures that all features, including one-hot encoded categorical variables, have been correctly transformed.
* This step confirms that the dataset is ready for machine learning model training.

In [40]:
X_train.shape

(3774, 45)

## Inspect Feature Names from DictVectorizer

In this step, we display the **feature names** generated by the fitted `DictVectorizer`:

* `dv.get_feature_names()` returns a list of all columns in the transformed feature matrix.
* This includes **one-hot encoded categorical features** and the original numerical features.
* Inspecting the feature names helps verify that the encoding was performed correctly and provides insight into the structure of the input for machine learning models.


In [41]:
dv.get_feature_names()

['contract=month-to-month',
 'contract=one_year',
 'contract=two_year',
 'dependents=no',
 'dependents=yes',
 'deviceprotection=no',
 'deviceprotection=no_internet_service',
 'deviceprotection=yes',
 'gender=female',
 'gender=male',
 'internetservice=dsl',
 'internetservice=fiber_optic',
 'internetservice=no',
 'monthlycharges',
 'multiplelines=no',
 'multiplelines=no_phone_service',
 'multiplelines=yes',
 'onlinebackup=no',
 'onlinebackup=no_internet_service',
 'onlinebackup=yes',
 'onlinesecurity=no',
 'onlinesecurity=no_internet_service',
 'onlinesecurity=yes',
 'paperlessbilling=no',
 'paperlessbilling=yes',
 'partner=no',
 'partner=yes',
 'paymentmethod=bank_transfer_(automatic)',
 'paymentmethod=credit_card_(automatic)',
 'paymentmethod=electronic_check',
 'paymentmethod=mailed_check',
 'phoneservice=no',
 'phoneservice=yes',
 'seniorcitizen',
 'streamingmovies=no',
 'streamingmovies=no_internet_service',
 'streamingmovies=yes',
 'streamingtv=no',
 'streamingtv=no_internet_servic

## 15. Training logistic regression

## Import Logistic Regression Model

Here, we import `LogisticRegression` from **Scikit-learn**:

* Logistic Regression is a widely used algorithm for **binary classification tasks**, such as predicting whether a customer will churn (`0` or `1`).
* It models the probability of the target variable based on input features and is interpretable, making it suitable for business applications.
* This step prepares us to initialize and train the churn prediction model.


In [42]:
from sklearn.linear_model import LogisticRegression

## Train Logistic Regression Model

In this step, we initialize and train a **Logistic Regression** model:

* `solver='liblinear'` is chosen for efficient optimization on small datasets and binary classification tasks.
* `random_state=1` ensures reproducibility of the results.
* `model.fit(X_train, y_train)` trains the model using the numeric training feature matrix and corresponding target values.

After this step, the model has learned the relationships between features and the likelihood of customer churn.


In [43]:
model = LogisticRegression(solver='liblinear', random_state=1)
model.fit(X_train, y_train)

LogisticRegression(random_state=1, solver='liblinear')

## Prepare Validation Feature Matrix

In this step, we transform the validation dataset into a numeric feature matrix:

* `df_val[categorical + numerical].to_dict(orient='records')` converts the validation features into a **list of dictionaries**, compatible with `DictVectorizer`.
* `X_val = dv.transform(val_dict)` applies the same encoding learned from the training data to ensure consistency.
* The resulting `X_val` is ready for evaluating the trained Logistic Regression model on unseen data.


In [44]:
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

## Predict Churn Probabilities on Validation Set

In this step, we use the trained Logistic Regression model to calculate **predicted probabilities** of churn for each validation example:

* `model.predict_proba(X_val)` returns an array of probabilities for each class (0 = no churn, 1 = churn).
* These probabilities indicate the likelihood of each customer churning.
* Predicted probabilities are useful for **threshold-based classification**, ROC analysis, and understanding model confidence.


In [45]:
model.predict_proba(X_val)

array([[0.76509452, 0.23490548],
       [0.73114964, 0.26885036],
       [0.68055068, 0.31944932],
       ...,
       [0.94275132, 0.05724868],
       [0.3847724 , 0.6152276 ],
       [0.93872722, 0.06127278]])

## Extract Churn Probabilities for Validation Set

In this step, we focus on the predicted probabilities for the **churn class (1)**:

* `y_pred = model.predict_proba(X_val)[:, 1]` selects the probabilities corresponding to customers who might churn.
* This vector contains the likelihood of each validation customer churning.
* These probabilities are essential for evaluation metrics such as **ROC curves, AUC**, or applying custom classification thresholds.


In [46]:
y_pred = model.predict_proba(X_val)[:, 1]

## Inspect Predicted Churn Probabilities

Here, we display the predicted probabilities for the validation set:

* `y_pred` contains the likelihood of each customer churning, as estimated by the trained Logistic Regression model.
* Inspecting these values helps verify that predictions are generated correctly and provides insight into the model's confidence for each example.
* These probabilities can be further used to make binary predictions or evaluate model performance.


In [47]:
y_pred

array([0.23490548, 0.26885036, 0.31944932, ..., 0.05724868, 0.6152276 ,
       0.06127278])

## Convert Probabilities to Binary Predictions

In this step, we transform the predicted churn probabilities into **binary predictions**:

* `churn = y_pred > 0.5` applies a threshold of 0.5.

  * If probability > 0.5 → predict `1` (churn)
  * If probability ≤ 0.5 → predict `0` (no churn)
* This converts continuous probabilities into **discrete class labels**, which can be used for accuracy calculation and other classification metrics.


In [48]:
churn = y_pred > 0.5

## Calculate Model Accuracy on Validation Set

In this step, we measure how well the model's binary predictions match the true target values:

* `(y_val == churn).mean()` computes the **proportion of correct predictions**.
* This provides the **accuracy** of the Logistic Regression model on the validation set.
* Accuracy helps us understand the model's overall performance before further evaluation with other metrics like ROC-AUC or confusion matrix.


In [49]:
(y_val == churn).mean()

0.8016129032258065

## 16. Model interpretation

## Inspect Logistic Regression Intercept

Here, we examine the **intercept (bias) term** of the trained Logistic Regression model:

* `model.intercept_[0]` represents the base log-odds of churn when all feature values are zero.
* The intercept provides insight into the model's baseline prediction before considering the effect of individual features.
* Understanding the intercept can help in interpreting model outputs and probabilities.


In [50]:
model.intercept_[0]

-0.12198811467233629

## Inspect Logistic Regression Coefficients

In this step, we map the **trained model's coefficients** to their corresponding feature names:

* `dict(zip(dv.get_feature_names(), model.coef_[0].round(3)))` creates a dictionary showing the effect of each feature on the log-odds of churn.
* Positive coefficients indicate that higher values of the feature increase the likelihood of churn, while negative coefficients decrease it.
* This mapping helps interpret the model and understand which features are most influential in predicting customer churn.

In [51]:
dict(zip(dv.get_feature_names(), model.coef_[0].round(3)))

{'contract=month-to-month': 0.563,
 'contract=one_year': -0.086,
 'contract=two_year': -0.599,
 'dependents=no': -0.03,
 'dependents=yes': -0.092,
 'deviceprotection=no': 0.1,
 'deviceprotection=no_internet_service': -0.116,
 'deviceprotection=yes': -0.106,
 'gender=female': -0.027,
 'gender=male': -0.095,
 'internetservice=dsl': -0.323,
 'internetservice=fiber_optic': 0.317,
 'internetservice=no': -0.116,
 'monthlycharges': 0.001,
 'multiplelines=no': -0.168,
 'multiplelines=no_phone_service': 0.127,
 'multiplelines=yes': -0.081,
 'onlinebackup=no': 0.136,
 'onlinebackup=no_internet_service': -0.116,
 'onlinebackup=yes': -0.142,
 'onlinesecurity=no': 0.258,
 'onlinesecurity=no_internet_service': -0.116,
 'onlinesecurity=yes': -0.264,
 'paperlessbilling=no': -0.213,
 'paperlessbilling=yes': 0.091,
 'partner=no': -0.048,
 'partner=yes': -0.074,
 'paymentmethod=bank_transfer_(automatic)': -0.027,
 'paymentmethod=credit_card_(automatic)': -0.136,
 'paymentmethod=electronic_check': 0.175,


## Prepare a Subset of Features for Logistic Regression

In this step, we create a smaller feature set using only selected features: `contract`, `tenure`, and `totalcharges`:

* `train_dict_small = df_train[subset].to_dict(orient='records')` converts the subset into dictionary format for encoding.
* `DictVectorizer(sparse=False)` is initialized and fitted on the smaller dataset.
* `X_small_train = dv_small.transform(train_dict_small)` transforms the subset into a numeric feature matrix.
* `dv_small.get_feature_names()` lists the feature names created from the subset.

Using a smaller feature set helps experiment with simpler models and evaluate the contribution of


In [52]:
subset = ['contract', 'tenure', 'totalcharges']
train_dict_small = df_train[subset].to_dict(orient='records')
dv_small = DictVectorizer(sparse=False)
dv_small.fit(train_dict_small)

X_small_train = dv_small.transform(train_dict_small)

dv_small.get_feature_names()

['contract=month-to-month',
 'contract=one_year',
 'contract=two_year',
 'tenure',
 'totalcharges']

## Train Logistic Regression Model on Selected Feature Subset

In this step, we initialize and train a Logistic Regression model using only the selected features (`contract`, `tenure`, `totalcharges`):

* `solver='liblinear'` is suitable for binary classification tasks.
* `random_state=1` ensures reproducibility of results.
* `model_small.fit(X_small_train, y_train)` trains the model on the smaller numeric feature matrix and corresponding target values.

Training on a subset helps evaluate the predictive power of key features and can simplify model interpretation.


In [53]:
model_small = LogisticRegression(solver='liblinear', random_state=1)
model_small.fit(X_small_train, y_train)

LogisticRegression(random_state=1, solver='liblinear')

## Inspect Intercept of Subset Model

Here, we examine the **intercept (bias) term** of the Logistic Regression model trained on the selected features:

* `model_small.intercept_[0]` represents the base log-odds of churn when `contract`, `tenure`, and `totalcharges` are zero.
* Understanding this intercept helps interpret how the model starts predicting churn before considering the effect of the chosen features.


In [9]:
model_small.intercept_[0]

NameError: name 'model_small' is not defined

## Inspect Coefficients of Subset Model

In this step, we map the **coefficients of the Logistic Regression model trained on the selected features** to their corresponding feature names:

* `dict(zip(dv_small.get_feature_names(), model_small.coef_[0].round(3)))` creates a dictionary showing the effect of each feature on the log-odds of churn.
* Positive coefficients indicate that higher values of the feature increase the likelihood of churn, while negative coefficients decrease it.
* This mapping allows us to interpret the contribution of key features (`contract`, `tenure`, `totalcharges`) in predicting customer churn.


In [ ]:
dict(zip(dv_small.get_feature_names(), model_small.coef_[0].round(3)))

{'contract=month-to-month': 0.866,
 'contract=one_year': -0.327,
 'contract=two_year': -1.117,
 'tenure': -0.094,
 'totalcharges': 0.001}

## Prepare Validation Set for Subset Model

In this step, we transform the validation dataset using the selected feature subset:

* `df_val[subset].to_dict(orient='records')` converts the validation features into dictionary format compatible with `DictVectorizer`.
* `X_small_val = dv_small.transform(val_dict_small)` applies the encoding learned from the training subset.
* The resulting feature matrix is ready for evaluating the smaller Logistic Regression model on unseen data.


In [ ]:
val_dict_small = df_val[subset].to_dict(orient='records')
X_small_val = dv_small.transform(val_dict_small)

## Predict Churn Probabilities for Validation Set (Subset Model)

In this step, we extract the predicted probabilities for the **churn class (1)** using the smaller feature subset model:

* `y_pred_small = model_small.predict_proba(X_small_val)[:, 1]` selects the probabilities corresponding to churn.
* These probabilities indicate the likelihood of each validation customer churning based on `contract`, `tenure`, and `totalcharges`.
* They are essential for evaluating the model’s performance and applying classification thresholds.


In [ ]:
y_pred_small = model_small.predict_proba(X_small_val)[:, 1]

## 17. Using the model

## Create a Sample Customer for Prediction

In this step, we define a **single customer record** as a dictionary:

* The dictionary includes all relevant **categorical and numerical features** used in the model.
* This allows us to demonstrate how to make **churn predictions for an individual customer**.
* Using a single record is useful for testing, demonstrations, or real-time prediction scenarios.


In [58]:
customer = {
    'customerid': '8879-zkjof',
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'no',
    'dependents': 'no',
    'tenure': 41,
    'phoneservice': 'yes',
    'multiplelines': 'no',
    'internetservice': 'dsl',
    'onlinesecurity': 'yes',
    'onlinebackup': 'no',
    'deviceprotection': 'yes',
    'techsupport': 'yes',
    'streamingtv': 'yes',
    'streamingmovies': 'yes',
    'contract': 'one_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'bank_transfer_(automatic)',
    'monthlycharges': 79.85,
    'totalcharges': 3320.75,
}

## Predict Churn Probability for a Single Customer

In this step, we use the trained Logistic Regression model to predict the **churn probability** for an individual customer:

* `X_test = dv.transform([customer])` converts the customer dictionary into the numeric feature matrix.
* `model.predict_proba(X_test)[0, 1]` returns the probability that this customer will churn (class 1).
* This demonstrates how the model can be applied for **real-time or individual customer predictions**.


In [59]:
X_test = dv.transform([customer])
model.predict_proba(X_test)[0, 1]

0.07332577315357781

## Inspect Feature Vector of a Single Customer

Here, we display the **numeric feature vector** for the individual customer after transformation:

* `list(X_test[0])` shows all the values in the feature vector, including **one-hot encoded categorical variables** and original numerical features.
* Inspecting this vector helps verify that the customer data was correctly encoded and ready for prediction.
* This step ensures transparency in how input features are represented to the model.


In [60]:
print(list(X_test[0]))

[0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 79.85, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 41.0, 3320.75]


## Define Another Sample Customer for Prediction

In this step, we create a **second customer record** with different attributes:

* The dictionary includes all relevant categorical and numerical features for the model.
* This allows us to test how the model predicts churn for a customer with different characteristics.
* Such examples are useful for **scenario analysis** and understanding model behavior across diverse customer profiles.


In [61]:
customer = {
    'gender': 'female',
    'seniorcitizen': 1,
    'partner': 'no',
    'dependents': 'no',
    'phoneservice': 'yes',
    'multiplelines': 'yes',
    'internetservice': 'fiber_optic',
    'onlinesecurity': 'no',
    'onlinebackup': 'no',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'yes',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 85.7,
    'totalcharges': 85.7
}

## Predict Churn Probability for Another Customer

In this step, we use the trained Logistic Regression model to predict the **churn probability** for the second sample customer:

* `X_test = dv.transform([customer])` converts the customer dictionary into the numeric feature matrix.
* `model.predict_proba(X_test)[0, 1]` returns the likelihood that this customer will churn (class 1).
* This demonstrates how the model responds to different customer attributes and can be used for scenario analysis or individual predictions.


In [62]:
X_test = dv.transform([customer])
model.predict_proba(X_test)[0, 1]

0.8321638622459152